# Week 3
# Data Cleaning & Dataset Preparation

This notebook performs data cleaning based on the findings from Week 2.

Objectives:

- Remove duplicate records
- Handle invalid values
- Remove unnecessary columns
- Keep only business-relevant fields
- Export clean datasets for future analysis

## Part 1: Load Validated Dataset

Load the residential listing and sold datasets generated from Week 1.

These datasets were validated during Week 2 and will now be cleaned for future analysis.

In [ ]:
import pandas as pd
import numpy as np

listings = pd.read_csv(
    "../outputs/combined_listings_residential.csv",
    low_memory=False
)

sold = pd.read_csv(
    "../outputs/combined_sold_residential.csv",
    low_memory=False
)

print(listings.shape)
print(sold.shape)

## Part 2: Remove Duplicate Columns

Some listing files contain duplicated columns with a `.1` suffix.

These duplicated fields are removed to ensure a consistent schema.

In [ ]:
duplicate_cols = [
    col for col in listings.columns
    if col.endswith(".1")
]

listings = listings.drop(columns=duplicate_cols)

print("Listings:", listings.shape)

## Part 3: Remove High-Missing Columns

According to the updated project guideline, columns with more than 90% missing values can be removed because they are unlikely to contribute meaningful information to the final Market Analysis and Competitive Analysis dashboards.

In [ ]:
listing_missing = listings.isnull().mean() * 100
sold_missing = sold.isnull().mean() * 100

listing_drop_cols = listing_missing[
    listing_missing > 90
].index

sold_drop_cols = sold_missing[
    sold_missing > 90
].index

listings = listings.drop(columns=listing_drop_cols)
sold = sold.drop(columns=sold_drop_cols)

print("Listings:", listings.shape)
print("Sold:", sold.shape)


Columns with more than 90% missing values were removed to improve data quality while preserving fields that are useful for the final dashboards.

## Part 4: Remove Duplicate Records

Duplicate rows were identified during Week 2.

They are now removed to eliminate redundant transactions.

In [ ]:
listing_before = len(listings)
sold_before = len(sold)

listings = listings.drop_duplicates()
sold = sold.drop_duplicates()

print(f"Listings: {listing_before} → {len(listings)}")
print(f"Sold: {sold_before} → {len(sold)}")

## Part 5: Handle Invalid Numeric Values

Several numeric fields contain invalid values such as zero prices, negative days on market, and zero living areas.

These values are converted to missing values for future analysis.

In [ ]:
sold.loc[
    sold["ClosePrice"] <= 0,
    "ClosePrice"
] = np.nan

sold.loc[
    sold["OriginalListPrice"] <= 0,
    "OriginalListPrice"
] = np.nan

sold.loc[
    sold["LivingArea"] <= 0,
    "LivingArea"
] = np.nan

sold.loc[
    sold["DaysOnMarket"] < 0,
    "DaysOnMarket"
] = np.nan

In [ ]:
print(sold[[
    "ClosePrice",
    "OriginalListPrice",
    "LivingArea",
    "DaysOnMarket"
]].isnull().sum())

## Part 6: Review Extreme Outliers

Extreme values were reviewed using the 99th percentile.

Outliers are retained for now because luxury properties may represent valid observations.

In [ ]:
for col in [
    "ClosePrice",
    "LivingArea",
    "DaysOnMarket"
]:

    threshold = sold[col].quantile(.99)

    print(col)

    print(
        sold[
            sold[col] > threshold
        ][col].describe()
    )

## Part 7: Feature  Inventory

Before cleaning the dataset, all available fields are reviewed.

For each column, we summarize:

- Data type
- Missing percentage
- Business meaning
- Whether it supports the final dashboard
- Keep / Drop decision

In [ ]:
feature_summary = pd.DataFrame({
    "Column": sold.columns,
    "Data Type": sold.dtypes.astype(str).values,
    "Missing %": (sold.isnull().mean()*100).round(2).values,
    "Missing Count": sold.isnull().sum().values,
    "Unique Values": sold.nunique().values,
    "Example Value": sold.iloc[0].values
})

feature_summary

In [ ]:
feature_summary.to_csv(
    "../outputs/feature_dictionary.csv",
    index=False
)

In [ ]:
feature_dictionary = pd.read_excel(
    "../outputs/feature_dictionary_filled.xlsx"
)

feature_dictionary.head()

drop_columns = feature_dictionary.loc[
    feature_dictionary["Decision"] == "Drop",
    "Column"
].tolist()

print(drop_columns)

In [ ]:
listings = listings.drop(
    columns=[c for c in drop_columns if c in listings.columns],
    errors="ignore"
)

sold = sold.drop(
    columns=[c for c in drop_columns if c in sold.columns],
    errors="ignore"
)

print("Listings:", listings.shape)
print("Sold:", sold.shape)

### Results

The following columns were removed from the working datasets:

- **OriginatingSystemName**
- **OriginatingSystemSubName**
- **BuyerAgencyCompensationType**
- **BuyerAgencyCompensation**

These fields were excluded because they are either:

- System metadata that does not contribute to business analysis, or
- Variables with more than **90% missing values**, making them unsuitable for the final Market Analysis and Competitive Analysis dashboards.

The dataset is now cleaner and contains only features that are more relevant for downstream analysis and dashboard development.

## Part 8: Export Clean Dataset

The cleaned datasets are exported for subsequent market analysis and dashboard development.

In [ ]:
listings.to_csv(
    "../outputs/clean_listings.csv",
    index=False
)

sold.to_csv(
    "../outputs/clean_sold.csv",
    index=False
)

print("Clean datasets exported.")

# Part 9: Mortgage Rate Enrichment

To support downstream market analysis, the cleaned MLS datasets are enriched with the national 30-year fixed mortgage rate published by the Federal Reserve (FRED).

Because mortgage rates are reported weekly while MLS transactions are analyzed monthly, the weekly observations are first aggregated into monthly averages before merging.

In [ ]:
import pandas as pd

url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"

mortgage = pd.read_csv(url)

print(mortgage.head())
print(mortgage.columns)

In [ ]:
import pandas as pd

url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"

mortgage = pd.read_csv(
    url,
    parse_dates=["observation_date"]
)

# Rename columns
mortgage.columns = [
    "date",
    "rate_30yr_fixed"
]

mortgage.head()

In [ ]:
mortgage["year_month"] = mortgage["date"].dt.to_period("M")

mortgage_monthly = (
    mortgage
    .groupby("year_month")["rate_30yr_fixed"]
    .mean()
    .reset_index()
)

mortgage_monthly.head()

In [ ]:
sold["year_month"] = (
    pd.to_datetime(
        sold["CloseDate"]
    ).dt.to_period("M")
)

listings["year_month"] = (
    pd.to_datetime(
        listings["ListingContractDate"]
    ).dt.to_period("M")
)

In [ ]:
sold = sold.merge(
    mortgage_monthly,
    on="year_month",
    how="left"
)

listings = listings.merge(
    mortgage_monthly,
    on="year_month",
    how="left"
)

In [ ]:
print(
    "Missing mortgage rates (Sold):",
    sold["rate_30yr_fixed"].isnull().sum()
)

print(
    "Missing mortgage rates (Listings):",
    listings["rate_30yr_fixed"].isnull().sum()
)

In [ ]:
sold[
    ["CloseDate","year_month","rate_30yr_fixed"]
].head()

listings[
    ["ListingContractDate","year_month","rate_30yr_fixed"]
].head()

### Results

The mortgage rate was successfully merged into both datasets.

The new feature (`rate_30yr_fixed`) will be available for downstream market trend analysis and dashboard development.

In [ ]:
sold.to_csv(
    "../outputs/clean_sold_with_rates.csv",
    index=False
)

listings.to_csv(
    "../outputs/clean_listings_with_rates.csv",
    index=False
)